<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Gradio UI für Agenten
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Die UI des PMO-Assistenten ist kein normales Chat-Fenster — sie muss Quellen, Trace-Informationen, Sicherheitseinschätzung und Freigabe-Status sichtbar machen, nicht nur den Antworttext. Eine Oberfläche, die Kontrolle verschweigt, untergräbt die Leitplanken aus der Leitaufgabe, egal wie gut das System dahinter ist.

**Praxis-Transfer:** Mara Vogt stellt ihre Briefing-Frage über eine einfache Oberfläche und sieht neben der Antwort auch Quellen, Tool-Aufrufe und Unsicherheit.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "false"
from langsmith import tracing_context
# Tracing bleibt standardmäßig aus; ausgewählte Referenzläufe werden explizit aktiviert.
os.environ["LANGSMITH_PROJECT"]  = "m24_gradio_ui"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGCHAIN_CALLBACKS_BACKGROUND"] = "false"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Colab stellt Gradio bereit; nur die API-Kompatibilität wird ermittelt.
import gradio as gr
GR_MESSAGES = (
    {'type': 'messages'}
    if int(gr.__version__.split('.')[0]) < 6
    else {}
)

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    WORKER,
    CODING,
    JUDGE,
    PLANNER,
    WORKER_PREMIUM,
    FRONTIER,
    IMAGE_GENERATION,
    TRANSCRIPTION,
    TRANSCRIPTION_SEGMENTS,
    EMBEDDINGS,
)
# LangSmith Tracing
run_cfg = {
    "tags": ["gradio"],
    "metadata": {"notebook": "gradio_ui_fuer_agenten", "version": "1.0"}
}

# 1 | Übersicht
---


Gradio macht den PMO-Assistenten interaktiv nutzbar: Fragen werden gestellt, Quellen werden sichtbar, Unsicherheit wird angezeigt und kritische Antworten können vor der Ausgabe freigegeben werden.

| Gradio-Komponente | Einsatz im PMO-Assistenten |
|-------------------|-------------------------------|
| `gr.ChatInterface` | Schnellste Chat-UI für Fragen an den Korpus |
| `gr.Blocks` | Chat, Quellenpanel und Tool-Log in einem Layout |
| `gr.State` | Sitzungsstatus, ausstehende Freigaben, Verlauf |
| Generator / `yield` | Fortschrittsschritte beim Retrieval anzeigen |
| Buttons | Human-in-the-Loop: Freigeben / Ablehnen |

# 2 | gr.ChatInterface — Briefing-Chat
---


`gr.ChatInterface` ist die schnellste Variante: eine Python-Funktion `fn(message, history) -> str` wird automatisch als Chat-UI gerendert.

```python
import gradio as gr

def chat_fn(message: str, history: list) -> str:
    return briefing_agent.invoke(...)

gr.ChatInterface(fn=chat_fn).launch()
```

| Parameter | Bedeutung |
|-----------|----------|
| `fn` | Callback-Funktion `(message, history) → str` |
| `title` | Überschrift der UI |
| `examples` | Klickbare Beispiel-Fragen |
| `type="messages"` | Nachrichtenformat für Gradio 5; bei Gradio 6 automatisch aktiv |
| `share=True` | Öffentliche URL für Colab/Remote |

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.agents import create_agent

llm = init_chat_model(WORKER)

# Wissensdatenbank aus RAG-Agent (Google Drive) — kein eigener Aufbau
from google.colab import drive
from genai_lib.pmo_rag import get_pmo_vectorstore

drive.mount("/content/drive")
vectorstore = get_pmo_vectorstore()
if vectorstore._collection.count() == 0:
    raise RuntimeError("Die Collection pmo_korpus ist leer. Bitte zuerst RAG-Agent ausführen.")

# Distanz-Schwellen (Chroma: kleiner = ähnlicher) — bei Bedarf an den Korpus anpassen
SCHWELLE_HOCH = 0.9
SCHWELLE_NICHT_IM_KORPUS = 1.3

def _waehle_snippet(frage: str) -> dict:
    """Holt den besten Treffer aus der Wissensdatenbank und leitet die Sicherheit aus der Distanz ab."""
    treffer = vectorstore.similarity_search_with_score(frage, k=1)
    if not treffer or treffer[0][1] > SCHWELLE_NICHT_IM_KORPUS:
        return {
            "quelle": "Nicht im Korpus",
            "passage": "Keine passende Passage in der Wissensdatenbank gefunden.",
            "sicherheit": "niedrig",
        }
    doc, distanz = treffer[0]
    return {
        "quelle": doc.metadata.get("source", "unbekannt").split("/")[-1],
        "passage": doc.page_content.replace("\n", " ")[:300],
        "sicherheit": "hoch" if distanz <= SCHWELLE_HOCH else "mittel",
    }

@tool
def quellen_suchen(frage: str) -> str:
    """Sucht die passendste Quelle in der Wissensdatenbank des Projekt-Korpus."""
    snippet = _waehle_snippet(frage)
    return f"Quelle: {snippet['quelle']}\nPassage: {snippet['passage']}\nSicherheit: {snippet['sicherheit']}"

@tool
def quellenabdeckung_pruefen(antwort: str) -> str:
    """Prüft, ob eine Antwort eine Quelle oder Nicht-im-Korpus-Hinweis enthält."""
    lower = antwort.lower()
    if "quelle:" in lower or "nicht im korpus" in lower:
        return "Quellenbindung: ok"
    return "Quellenbindung: fehlt"

tools_liste = [quellen_suchen, quellenabdeckung_pruefen]
briefing_agent = create_agent(
    model=llm,
    tools=tools_liste,
    system_prompt=(
        "Rolle: PMO-Assistent für Projekt Kompass. "
        "Antworten brauchen Quelle, Passage und Sicherheit. "
        "Wenn keine Passage vorhanden ist: Nicht im Korpus schreiben."
    ),
)
print("✅ PMO-Assistent mit Quellen- und Prüf-Tool bereit")

In [ ]:
with tracing_context(enabled=True):
    import gradio as gr

    # UI-Interaktionen bleiben standardmäßig ungetraced.


    def chat_fn(message: str, history: list) -> str:
        """Gradio-Callback: gibt eine quellengebundene Briefing-Antwort zurück."""
        result = briefing_agent.invoke(
            {"messages": [HumanMessage(content=message)]},
            config={"run_name": "m24_kap2_basicbriefingchat", "tags": ["basic-chat", "briefing"]},
        )
        return result["messages"][-1].content


    basic_probe = chat_fn("Was wurde am 17.03.2026 zur Vektordatenbank entschieden?", [])
    print(basic_probe[:500])

    demo_basic = gr.ChatInterface(
        fn=chat_fn,
        title="PMO-Assistent",
        description="Fragen an den Projekt-Korpus mit Quelle, Passage und Sicherheit.",
        examples=[
            "Was wurde am 17.03.2026 zur Vektordatenbank entschieden?",
            "Welchen Status hat Risiko R2?",
            "Was hat sich am 14.04.2026 geändert?",
        ],
        **GR_MESSAGES,
    )
    demo_basic.launch(quiet=True)


# 3 | gr.Blocks — Chat mit Quellen- und Tool-Visualisierung
---


`gr.Blocks` erlaubt ein Layout mit mehreren Panels. Hier: Chat links, Quellen und Tool-Log rechts.

```
┌─────────────────────────┬────────────────────────────┐
│  Chat                   │  Quellen / Tool-Aufrufe    │
│  Frage an den Korpus    │  quellen_suchen(...)        │
│  Antwort                │  Quelle + Sicherheit       │
└─────────────────────────┴────────────────────────────┘
```


In [ ]:
import gradio as gr


def verarbeite(nachricht: str, verlauf: list):
    """Führt den PMO-Assistenten aus und extrahiert Tool-Aufrufe für den Log."""
    verlauf = verlauf or []
    result = briefing_agent.invoke(
        {"messages": [HumanMessage(content=nachricht)]},
        config={"run_name": "m24_kap3_blocksbriefingchat", "tags": ["blocks", "briefing"]},
    )

    tool_log_lines = []
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                args_str = ", ".join(f"{k}={v}" for k, v in tc["args"].items())
                tool_log_lines.append(f"▶ {tc['name']}({args_str})")
        if isinstance(msg, ToolMessage):
            tool_log_lines.append(f"↳ {msg.content}")

    antwort = result["messages"][-1].content
    verlauf.append({"role": "user", "content": nachricht})
    verlauf.append({"role": "assistant", "content": antwort})
    tool_text = "\n".join(tool_log_lines) if tool_log_lines else "(keine Tools aufgerufen)"
    return "", verlauf, tool_text


blocks_probe = verarbeite("Welchen Status hat Risiko R2?", [])
print(blocks_probe[2][:500])

with gr.Blocks(title="PMO-Assistent mit Quellen", theme=gr.themes.Soft()) as demo_blocks:
    gr.Markdown("## PMO-Assistent mit Quellenpanel")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(**GR_MESSAGES, label="Chat", height=300)
        with gr.Column(scale=2):
            tool_box = gr.Textbox(
                label="Quellen und Tool-Log", lines=12,
                interactive=False, placeholder="Noch keine Quellen geprüft..."
            )
    with gr.Row():
        eingabe = gr.Textbox(placeholder="Frage an den Projekt-Korpus", label="Eingabe", scale=4)
        senden = gr.Button("Senden", variant="primary", scale=1)

    senden.click(verarbeite, [eingabe, chatbot], [eingabe, chatbot, tool_box])
    eingabe.submit(verarbeite, [eingabe, chatbot], [eingabe, chatbot, tool_box])

demo_blocks.launch(quiet=True)


# 4 | Streaming — Suchfortschritt anzeigen
---


LangGraph unterstützt `stream()` — damit lassen sich Zwischenschritte an Gradio weiterleiten. Für Briefing-Workflows ist oft wichtiger, den Fortschritt sichtbar zu machen: Anfrage prüfen, Quellen suchen, Antwort formulieren.

> **Hinweis:** `stream()` liefert Chunks pro Node — nicht zwingend pro Token. Für echtes Token-Streaming ist `astream_events()` nötig.


In [ ]:
#@markdown   <p><font size="4" color='green'>  Streaming-Flow</font> </br></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
sequenceDiagram
    autonumber
    participant U as 👤 User
    participant G as gr.ChatInterface
    participant LG as LangGraph
    participant T as Tools

    U->>G: Frage
    G->>LG: stream({"messages": [...]})
    loop Chunk für Chunk
        LG-->>G: {"agent": {"messages": [...]}}
        G-->>U: yield partial_antwort
    end
    LG->>T: Tool Call
    T-->>LG: Ergebnis
    LG-->>G: {"agent": {"messages": [finale Antwort]}}
    G-->>U: yield finale_antwort
'''
mermaid(diagram, width=900)

In [ ]:
import gradio as gr


def stream_agent(message: str, history: list):
    """Generator: zeigt Suchfortschritt und finale Antwort."""
    yield "Anfrage wird geprüft..."
    yield "Quellen werden gesucht..."
    antwort = chat_fn(message, history)
    yield antwort


stream_probe = list(stream_agent("Welchen Status hat Risiko R2?", []))
print(stream_probe[-1][:500])

demo_stream = gr.ChatInterface(
    fn=stream_agent,
    title="PMO-Assistent Streaming",
    description="Suchfortschritt und finale quellengebundene Antwort.",
    examples=["Welchen Status hat Risiko R2?", "Was wurde am 17.03.2026 entschieden?"],
    **GR_MESSAGES,
)
demo_stream.launch(quiet=True)


# 5 | Human-in-the-Loop UI
---


Das HITL-Muster bekommt hier eine Oberfläche für Freigabe. Der PMO-Assistent bereitet eine Antwort mit Quelle und Sicherheit vor. Bei niedriger Sicherheit bleibt die finale Ausgabe gesperrt, bis eine Freigabe erfolgt.

```
Frage → Antwortentwurf + Quelle + Sicherheit
          ├─ Sicherheit hoch/mittel → anzeigen
          └─ Sicherheit niedrig     → Freigabe erforderlich
```

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# Checkpointer bleibt sichtbar, weil UI-Sessions und Freigaben Zustand brauchen.
hitl_checkpointer = InMemorySaver()
print("✅ HITL-Session-Checkpointer bereit")


In [ ]:
#@markdown   <p><font size="4" color='green'>  Freigabe bei niedriger Sicherheit</font> </br></p>
hitl_diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart LR
    Q["Frage"] --> D["Antwortentwurf"]
    D --> S{"Sicherheit niedrig?"}
    S -->|nein| A["Antwort anzeigen"]
    S -->|ja| H["Freigabe erforderlich"]
    H -->|Genehmigen| A
    H -->|Ablehnen| R["Nicht freigegeben"]

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    classDef stop fill:#fee2e2,stroke:#dc2626,stroke-width:2px,color:#7f1d1d;

    %% Zuweisung
    class Q input;
    class S,H metric;
    class A output;
    class R stop;
'''
mermaid(hitl_diagram, width=800)


In [ ]:
import gradio as gr

sitzung = {"ausstehend": None}


def erstelle_entwurf(frage: str) -> dict:
    """Erstellt deterministisch einen Briefing-Entwurf mit Quelle und Sicherheit."""
    snippet = _waehle_snippet(frage)
    return {
        "frage": frage,
        "antwort": f"Antwort: {snippet['passage']}",
        "quelle": snippet["quelle"],
        "sicherheit": snippet["sicherheit"],
    }


def sende_anfrage(nachricht: str, verlauf: list):
    """Erstellt Entwurf und fordert bei niedriger Sicherheit Freigabe an."""
    verlauf = verlauf or []
    verlauf.append({"role": "user", "content": nachricht})
    entwurf = erstelle_entwurf(nachricht)
    text = (
        f"{entwurf['antwort']}\n\n"
        f"Quelle: {entwurf['quelle']}\n"
        f"Sicherheit: {entwurf['sicherheit']}"
    )
    if entwurf["sicherheit"] == "niedrig":
        sitzung["ausstehend"] = entwurf
        verlauf.append({"role": "assistant", "content": f"Freigabe erforderlich:\n\n{text}"})
        return "", verlauf, gr.update(visible=True)
    sitzung["ausstehend"] = None
    verlauf.append({"role": "assistant", "content": text})
    return "", verlauf, gr.update(visible=False)


def genehmige(verlauf: list):
    """Gibt einen ausstehenden Briefing-Entwurf frei."""
    pending = sitzung.get("ausstehend")
    if not pending:
        return verlauf, gr.update(visible=False)
    verlauf.append({"role": "assistant", "content": f"Freigegeben:\n\n{pending['antwort']}\nQuelle: {pending['quelle']}"})
    sitzung["ausstehend"] = None
    return verlauf, gr.update(visible=False)


def lehne_ab(verlauf: list):
    """Lehnt einen ausstehenden Briefing-Entwurf ab."""
    sitzung["ausstehend"] = None
    verlauf.append({"role": "assistant", "content": "Nicht freigegeben. Keine finale Ausgabe."})
    return verlauf, gr.update(visible=False)


hitl_probe = erstelle_entwurf("Frage außerhalb des Korpus")
print(hitl_probe)

with gr.Blocks(title="Briefing HITL", theme=gr.themes.Soft()) as demo_hitl:
    gr.Markdown("## PMO-Assistent mit Freigabe")
    chatbot = gr.Chatbot(**GR_MESSAGES, label="Chat", height=350)
    with gr.Row():
        eingabe = gr.Textbox(placeholder="Frage an den Projekt-Korpus", label="Eingabe", scale=4)
        senden_btn = gr.Button("Senden", variant="primary", scale=1)
    with gr.Row(visible=False) as approval_row:
        approve_btn = gr.Button("Freigeben", variant="primary")
        reject_btn = gr.Button("Ablehnen", variant="stop")

    senden_btn.click(sende_anfrage, [eingabe, chatbot], [eingabe, chatbot, approval_row])
    eingabe.submit(sende_anfrage, [eingabe, chatbot], [eingabe, chatbot, approval_row])
    approve_btn.click(genehmige, [chatbot], [chatbot, approval_row])
    reject_btn.click(lehne_ab, [chatbot], [chatbot, approval_row])

demo_hitl.launch(quiet=True)


In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace(os.environ["LANGSMITH_PROJECT"], limit=3, show_steps=True)


# A | Aufgaben
---


<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**

Eine eigene Briefing-Chat-UI mit mindestens zwei Tools bauen: Quellen suchen und Quellenbindung prüfen.


**Aufbau**

Eine `gr.Blocks`-App mit Chat, Quellenpanel und Tool-Log bauen.


**Vertiefung**

Streaming und Human-in-the-Loop kombinieren: Fortschritt anzeigen und bei niedriger Sicherheit eine manuelle Freigabe verlangen.


<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [KI-Agent](https://ralf-42.github.io/Agenten/visualisierungen/ki_agenten/)
- [LangGraph](https://ralf-42.github.io/Agenten/visualisierungen/langgraph_explainer/)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [PMO-Assistent im Betrieb](https://ralf-42.github.io/Agenten/08-deployment-betrieb/pmo-assistent-praxis.html)
- [Tool Use & Function Calling](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/tool-use-function-calling.html)
- [Einsteiger LangGraph](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-langgraph.html)
- [Human-in-the-Loop](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/human-in-the-loop.html)
- [Aus Entwicklung ins Deployment](https://ralf-42.github.io/Agenten/08-deployment-betrieb/aus-entwicklung-ins-deployment.html)
